# Expectations about inflation, house prices, and the stock market

This notebook analyses household expectations in the United States using the
[Survey of Consumer Expectations (SCE)](https://www.newyorkfed.org/microeconomics/sce#/),
run by the Federal Reserve Bank of New York.

In [1]:
from pathlib import Path

import pandas as pd

---
## Part 1 — Reading in the data

Each survey wave is stored in its own file named `SCE-MMM-YYYY.csv`, where each
row is one respondent in that month.

### 1.1 Combining the survey files

All SCE files have the same columns, so they are stacked on top of each other.
Three features of the data shape how they are read:

- The files are separated by semicolons, not commas.
- `data/` also contains `CPI.csv`, so the search only matches `SCE-*.csv`.
- Filenames sort alphabetically by month rather than by date, so the combined
  data is sorted by date afterwards.

In [2]:
def load_sce_data(data_dir='data'):
    """
    Read all SCE survey files and stack them into a single DataFrame.

    Parameters
    ----------
    data_dir : str or Path
        Folder containing the ``SCE-MMM-YYYY.csv`` files.

    Returns
    -------
    pandas.DataFrame
        One row per respondent and survey wave, sorted by date and respondent.
    """
    # Only match survey files, which excludes CPI.csv in the same folder
    files = sorted(Path(data_dir).glob('SCE-*.csv'))
    frames = [pd.read_csv(file, sep=';', parse_dates=['date']) for file in files]

    df = pd.concat(frames, ignore_index=True)

    return df.sort_values(['date', 'userid'], ignore_index=True)

In [3]:
df = load_sce_data()

### 1.2 Sample statistics

The function below is also used in Part 2 to report the same statistics for the
final data set.

In [4]:
def compute_sample_stats(df):
    """
    Compute summary statistics describing the size of a sample.

    Parameters
    ----------
    df : pandas.DataFrame
        Survey data with the columns ``userid``, ``wid`` and ``date``.

    Returns
    -------
    pandas.DataFrame
        One row per statistic, with the result in the column ``Value``.
    """
    stats = {
        'Unique individuals': df['userid'].nunique(),
        'Observations (rows)': len(df),
        'Unique survey waves': df['wid'].nunique(),
        'First date': df['date'].min().date(),
        'Last date': df['date'].max().date(),
    }

    return pd.Series(stats, name='Value').to_frame()

In [5]:
compute_sample_stats(df)

,Value
Unique individuals,23369
Observations (rows),176101
Unique survey waves,139
First date,2013-06-01
Last date,2024-12-31


The combined data set contains 176,101 observations from 23,369 individuals
across 139 monthly survey waves. On average, this is about 7.5 observations per
individual, which is below the usual 12 because some respondents leave the panel
early and others are still in it when the sample ends.

The data runs from June 2013 to December 2024, so it starts earlier than the 2014
launch date and the 2015–2024 analysis period given in the assignment.